## 行锁
```sql
BEGIN;

SELECT balance
FROM accounts
WHERE id = 1
FOR UPDATE;
```
`FOR UPDATE`的意思是：我读取这条数据并准备修改它，在我的事务结束前，其他事务不要修改或锁定这一行。
### 事务b普通查询
```sql
SELECT balance
FROM accounts
WHERE id = 1;
```
通常可以立即执行，因为MVCC允许事务B读取一个对自己可见的已提交版本。

所以行锁不等于禁止所有人读取。

## 并发扣款与悲观锁
### 使用悲观锁
```sql
BEGIN;

SELECT balance
FROM accounts
WHERE id = 1
FOR UPDATE;

-- 应用代码判断余额是否充足

UPDATE accounts
SET balance = balance - 80
WHERE id = 1;

COMMIT;
```
- A 锁定账户 1，读取余额 100
- B 尝试锁定账户 1 → 等待
- A 扣除 80，提交，余额变为 20
- B 获得锁，读取到最新余额 20
- B 判断扣除 50 不足 → 拒绝扣款

#### 悲观锁不适合所有场景
适合：
- 库存扣减、余额扣款、抢占唯一资源
不适合长时间持锁，例如：
- 锁定数据
- 等待用户付款
- 调用 20 秒的外部 API
- 再更新数据

## 死锁
### 一个转账死锁
2个转账同时执行：
- 事务A:账户1转给账户2
- 事务B:账户2转给账户1
2个都锁住了，这就是死锁。

### PostgreSQL 的处理方式
PostgreSQL 会检测死锁，并主动中止其中一个事务：
- 事务A：继续执行
- 事务B：收到deadlock_detected错误,并回滚
```
捕获死锁错误
→ 回滚当前事务
→ 从头重试整个事务

不是从失败的那条 SQL 接着重试，因为事务的上下文可能已经不可靠。
```

#### 最好的预防：统一加锁顺序
无论是从 1 转 2，还是从 2 转 1，都按账户 ID 小到大锁：
- 先锁 min(from_id, to_id)
- 再锁 max(from_id, to_id)

等待不一定是死锁；形成等待环才是死锁。

## 乐观锁与重试
- 悲观锁：先锁，再修改；冲突者等待。
- 乐观锁：不先锁；更新时校验版本号；冲突者更新 0 行。
- 
例如文档当前是：
`id=1, title="v1", version=3`
A 和 B 同时读到 `version=3`。A 先提交后，版本变为 4。
B 再执行：
```
UPDATE documents
SET title = 'B 的修改',
    version = version + 1
WHERE id = 1
  AND version = 3;
```
由于版本已不是 3，结果是：更新 0 行